<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L60_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L60: Capstone Project - End-to-End LLM Application

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 60 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Integrate concepts from the full course: data, model, training, evaluation, and deployment
- Build one production-ready pipeline: e.g. domain RAG + fine-tuned generator + API
- Meet comprehensive requirements for a complete LLM system

---

## Concept: Capstone

**Capstone** = end-to-end application using L1–L59: data prep (L37), optional tokenizer (L38), model choice and fine-tuning (L16–L18, L24), RAG (L28–L29), evaluation (L14, L54), security (L55), and deployment (L52). We wire a single pipeline: RAG + generator + simple API structure.

---

In [ ]:
!pip install transformers torch sentence-transformers faiss-cpu -q
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import faiss
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Index (RAG)

---

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
docs = ["LLMs are trained on large text corpora.", "RAG combines retrieval with generation.", "Fine-tuning adapts models to new tasks."]
emb = np.array(encoder.encode(docs)).astype("float32")
index = faiss.IndexFlatL2(emb.shape[1])
index.add(emb)
print("Index ready.")

## Part 2: Generator and RAG Query

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

def rag_query(query, top_k=2, max_new=30):
    q_emb = np.array(encoder.encode([query])).astype("float32")
    _, idx = index.search(q_emb, top_k)
    context = " ".join([docs[i] for i in idx[0]])
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
    out = generator(prompt, max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return out[0]["generated_text"][len(prompt):].strip()

answer = rag_query("What is RAG?")
print("Q: What is RAG?")
print("A:", answer)

## Part 3: Production Checklist

---

In [ ]:
checklist = [
    "[x] Data / index for RAG",
    "[x] Model loading and generation",
    "[x] RAG query function",
    "[ ] Fine-tune generator on domain (L24, L45)",
    "[ ] Evaluation (L54) and security review (L55)",
    "[ ] FastAPI or similar (L52), rate limits, monitoring",
]
for item in checklist:
    print(item)

## Capstone Requirements (Full)

- [x] Data preparation and retrieval (RAG)
- [x] Model: load and generate
- [ ] Training: optional fine-tuning or domain adaptation
- [ ] Evaluation: metrics on a held-out set
- [ ] Deployment: API, health checks, and docs
- [ ] Security: input/output checks, prompt injection awareness

---

## Exercises

1. Add a real document corpus, re-index, and run 10 QA pairs; report BLEU/ROUGE or match.
2. Wrap rag_query in FastAPI POST /query and test with curl.
3. Add a simple prompt injection test and document how your design mitigates it.

---

## Key Takeaways

1. End-to-end = data + model + retrieval + generation + eval + deploy + security.
2. Start minimal (as here), then add fine-tuning, eval, and production hardening.
3. The full course (L1–L60) supports every step of this pipeline.

---

## Congratulations

You have completed the **LLM Mastery** course (60 lessons). You can now design, train, evaluate, and deploy LLM systems from basics to expert level.

---